# Layer normalization notes

### Motivation

Batch gradient descent
- Feed the entire dataset through to calculate loss on the entire set.

Stochastic gradient descent
- Choose random minibatches to estimate the gradient across the entire set instead of one sample at a time.
  - Pros: Increased training speed, takes better advantage of parallelism, escape local minima
  - Cons: Introduces more complexity, more hyperparameters etc.

Batch normalization (https://arxiv.org/pdf/1502.03167)
- Internal Covariate Shift: change in the distribution of network activations due to the change in network parameters during training. For example, the parameters may send activations far out to the saturating regions of a sigmoid (resulting in vanishing gradients; typically solved with things like ReLU/init/learning rates but still something to be wary of)
- Problem: Essentially we'd like to make sure the inputs to nonlinearities are unit gaussian, which is most important for saturating nonlinearities like tanh, but also still important for others like ReLU to avoid neurons dying/being fully linear. This is to promote less "internal covariate shift" or "loss landscape smoothness"
- Solution: Add learnable layers to the network architecture itself:
  - Over a minibatch $x=\{x_1,\dots,x_m\}$ each of size $C$ (number of features),
  - Normalize it per-feature over the minibatch i.e. over the batch dimension to mean=0, std=1, $\hat{x_i}$. Then the output is all of the $y_i=\gamma\odot\hat{x_i}+\beta$, where $\gamma,\beta\in\mathbb{R}^C$.
  - $\odot$ is the element-wise product.

- We get these nice properties:
  - We initialize at unit gaussian
  - The network is free to adjust these params during backpropagation
  - Everything is differentiable so backpropagation actually works
- Issues:
  - We are tying each training pass to the size of a batch, which means during eval time we can't straightforwardly run it, and need to do something like freeze a mean/std to use during test time, or keep a running total. This extra state is kinda a headache, and leads to differences between train and eval time modes.
  - Each training pass is dependent on other examples in the batch. This does actually introduce some potentially favorable jitter/regularization, it seems.

Layer normalization (https://arxiv.org/pdf/1607.06450)
- Problem: Same problem as before, plus wanting to avoid the dependence on the batch - for example in variable-length examples, or 1-length batches (during test time).
- Solution: The batch norm idea, where we add a learnable layer, but untied from the batch size:
  - Over a minibatch = $x=\{x_1,\dots,x_m\}$ each of size $C$ (number of features),
  - Normalize each $x_i$ independently using only its $C$ feature values i.e. over the **feature/embedding** dimension to mean=0, std=1, $\hat{x_i}$, and add the learnable parameters $y_i=\gamma\odot\hat{x_i}+\beta$ where $\gamma,\beta\in\mathbb{R}^C$.
  - $\odot$ is the element-wise product.

Layer normalization is necessary for transformers: (B, T, C) for a batch * time * channels (embedding dimensions) training example, we would be doing:
- A normalization across the channels/embedding dimension per (B, T)
  - This prevents the time dimension from mixing as well, which would allow reading from the future, which we do not want, and explicitly prevent with the mask in the self-attention step.
  - This also lets us work with batch sizes of 1 (which is what generation is).
- This is in contrast to a batch-norm which would normalize across the batch (and traditionally time).